# Etapa 1: curadoria

Entrada: `data/raw/adult.csv`, baixado do Kaggle (*Adult Census Income*, publicado por UCI Machine Learning, versão 3, de 07/10/2016, licença CC0). É a base *Adult* do repositório da UCI, extraída por Barry Becker do censo dos EUA de 1994: cada linha é uma pessoa adulta, e o alvo diz se a renda anual passa de 50 mil dólares.

Saída: `data/processed/renda.csv`, com os valores ausentes marcados como vazios, sem linhas repetidas e com o alvo em 0/1.

## 0. Configuração

In [1]:
import sys
import hashlib
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import config

sha256 = hashlib.sha256(config.RAW_CSV.read_bytes()).hexdigest().upper()
print("SHA-256 confere:", sha256 == config.RAW_SHA256)

bruto = pd.read_csv(config.RAW_CSV)
print(bruto.shape)
bruto.head()

SHA-256 confere: True
(32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## 1. Colunas e tipos

São 14 atributos e o alvo. Seis atributos são números e oito são texto (categorias).

In [2]:
pd.DataFrame({
    "nome": [config.NOMES[c] for c in bruto.columns],
    "tipo": bruto.dtypes.astype(str),
    "valores distintos": bruto.nunique(),
    "exemplo": bruto.iloc[0],
})

,nome,tipo,valores distintos,exemplo
age,idade,int64,73,90
workclass,tipo de vínculo,str,9,?
fnlwgt,peso amostral,int64,21648,77053
education,escolaridade (texto),str,16,HS-grad
education.num,nível de escolaridade,int64,16,9
marital.status,estado civil,str,7,Widowed
occupation,ocupação,str,15,?
relationship,papel na família,str,6,Not-in-family
race,raça,str,5,White
sex,sexo,str,2,Female


In [3]:
texto = bruto.select_dtypes(exclude="number").columns
espacos = {c: int((bruto[c] != bruto[c].str.strip()).sum()) for c in texto}
print("valores com espaço no início ou no fim:", espacos)

valores com espaço no início ou no fim: {'workclass': 0, 'education': 0, 'marital.status': 0, 'occupation': 0, 'relationship': 0, 'race': 0, 'sex': 0, 'native.country': 0, 'income': 0}


Diferente da base de empréstimos, os textos já vêm sem espaços. A versão original da UCI tem um espaço antes de cada valor; a do Kaggle já foi limpa.

## 2. Valores ausentes

O CSV não tem células vazias: os valores ausentes vêm como o texto `?`.

In [4]:
print("células vazias:", int(bruto.isna().sum().sum()))
ausentes = (bruto == config.AUSENTE).sum()
ausentes = ausentes[ausentes > 0]
pd.DataFrame({"pessoas com ?": ausentes, "% da base": (100 * ausentes / len(bruto)).round(1)})

células vazias: 0


,pessoas com ?,% da base
workclass,1836,5.6
occupation,1843,5.7
native.country,583,1.8


In [5]:
sem_vinculo = bruto["workclass"] == config.AUSENTE
sem_ocupacao = bruto["occupation"] == config.AUSENTE
print("vínculo e ocupação ausentes juntos:", int((sem_vinculo & sem_ocupacao).sum()))
print("só a ocupação ausente:", int((~sem_vinculo & sem_ocupacao).sum()),
      "| vínculo nessas linhas:", bruto.loc[~sem_vinculo & sem_ocupacao, "workclass"].unique())
print("linhas com algum ?:", int((bruto == config.AUSENTE).any(axis=1).sum()))

acima = bruto[config.TARGET] == ">50K"
pd.DataFrame({
    "pessoas": [len(bruto), int(sem_vinculo.sum()), int((bruto["native.country"] == config.AUSENTE).sum())],
    "% com renda acima de 50 mil": [round(100 * acima.mean(), 1), round(100 * acima[sem_vinculo].mean(), 1),
                                    round(100 * acima[bruto["native.country"] == config.AUSENTE].mean(), 1)],
}, index=["base inteira", "vínculo ausente", "país ausente"])

vínculo e ocupação ausentes juntos: 1836
só a ocupação ausente: 7 | vínculo nessas linhas: <StringArray>
['Never-worked']
Length: 1, dtype: str
linhas com algum ?: 2399


,pessoas,% com renda acima de 50 mil
base inteira,32561,24.1
vínculo ausente,1836,10.4
país ausente,583,25.0


O vínculo e a ocupação faltam quase sempre juntos; a ocupação falta sozinha só para quem nunca trabalhou (`Never-worked`), o que faz sentido. Quem tem vínculo ausente ganha acima de 50 mil bem menos que a média: a ausência carrega informação, e apagar essas linhas mudaria a proporção do alvo.

**Decisão:** as linhas ficam, e o `?` vira célula vazia na base processada. Como tratar o vazio (por exemplo, como uma categoria própria, "ausente") é decisão do pré-processamento de cada modelo, dentro do `Pipeline`, e não da curadoria.

## 3. Linhas repetidas

In [6]:
repetidas = bruto[bruto.duplicated(keep=False)].sort_values(list(bruto.columns))
print("linhas repetidas (sem contar a primeira de cada grupo):", int(bruto.duplicated().sum()))
print("grupos:", repetidas.groupby(list(bruto.columns)).ngroups)
repetidas.head(6)

linhas repetidas (sem contar a primeira de cada grupo): 24
grupos: 23


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
19622,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
20507,19,Private,97261,HS-grad,9,Never-married,Farming-fishing,Not-in-family,White,Male,0,0,40,United-States,<=50K
10307,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K
22783,19,Private,138153,Some-college,10,Never-married,Adm-clerical,Own-child,White,Female,0,0,10,United-States,<=50K
17456,19,Private,146679,Some-college,10,Never-married,Exec-managerial,Own-child,Black,Male,0,0,30,United-States,<=50K
22934,19,Private,146679,Some-college,10,Never-married,Exec-managerial,Own-child,Black,Male,0,0,30,United-States,<=50K


As repetições são idênticas em todas as 15 colunas. Podem ser pessoas diferentes com os mesmos dados: o peso amostral `fnlwgt` se repete entre pessoas (21.648 valores distintos em 32.561 linhas), porque é calculado a partir de grupos demográficos. Não há como saber se são registros duplicados ou pessoas distintas.

**Decisão:** as repetições são removidas, mantendo a primeira de cada grupo. São 24 linhas (0,07% da base), e uma cópia no treino com outra no teste daria ao modelo um acerto sem mérito.

In [7]:
pares = pd.crosstab(bruto["education"], bruto["education.num"])
print("cada escolaridade tem um único código:", bool(((pares > 0).sum(axis=1) == 1).all()))
pares.idxmax(axis=1).sort_values().to_frame("education.num")

cada escolaridade tem um único código:

 True


,education.num
education,
Preschool,1
1st-4th,2
5th-6th,3
7th-8th,4
9th,5
10th,6
11th,7
12th,8
HS-grad,9


In [8]:
print(bruto["fnlwgt"].describe().round(0).to_string())
print("\ncorrelação de fnlwgt com o alvo:", round(bruto["fnlwgt"].corr(acima.astype(int)), 3))

count      32561.0
mean      189778.0
std       105550.0
min        12285.0
25%       117827.0
50%       178356.0
75%       237051.0
max      1484705.0

correlação de fnlwgt com o alvo: -0.009


- **`education`** é o mesmo que `education.num` em texto: cada nível de escolaridade tem um único código, de 1 (`Preschool`) a 16 (`Doctorate`). Fica só `education.num`, que já traz a ordem dos níveis.
- **`fnlwgt`** (*final weight*) é o peso amostral do censo: quantas pessoas da população cada registro representa. É uma propriedade da amostragem, não da pessoa, e quase não se relaciona com o alvo.

**Decisão:** as duas colunas ficam na base processada, para rastreio, mas não entram como atributos (`config.FORA`). Os modelos usam 12 atributos: 5 numéricos e 7 categóricos (`config.FEATURES`).

## 5. Valores extremos

In [9]:
print(bruto[config.NUMERICOS].describe().round(1).to_string())
print("\nganho de capital = 99.999:", int((bruto["capital.gain"] == 99999).sum()),
      "| renda acima de 50 mil entre eles:", round(100 * acima[bruto["capital.gain"] == 99999].mean(), 1), "%")
print("ganho de capital = 0:", round(100 * (bruto["capital.gain"] == 0).mean(), 1), "%",
      "| perda de capital = 0:", round(100 * (bruto["capital.loss"] == 0).mean(), 1), "%")
print("horas por semana = 99:", int((bruto["hours.per.week"] == 99).sum()))

           age  education.num  capital.gain  capital.loss  hours.per.week
count  32561.0        32561.0       32561.0       32561.0         32561.0
mean      38.6           10.1        1077.6          87.3            40.4
std       13.6            2.6        7385.3         403.0            12.3
min       17.0            1.0           0.0           0.0             1.0
25%       28.0            9.0           0.0           0.0            40.0
50%       37.0           10.0           0.0           0.0            40.0
75%       48.0           12.0           0.0           0.0            45.0
max       90.0           16.0       99999.0        4356.0            99.0

ganho de capital = 99.999: 159 | renda acima de 50 mil entre eles: 100.0 %
ganho de capital = 0: 91.7 % | perda de capital = 0: 95.3 %
horas por semana = 99: 85


O ganho de capital de 99.999 é o teto do censo (valores maiores foram cortados nesse número), e todas as 159 pessoas com esse valor têm renda acima de 50 mil: é um valor real, só truncado. Ganho e perda de capital são zero para a grande maioria das pessoas, o que torna as duas colunas muito assimétricas; isso importa para a padronização da SVM e fica para a análise exploratória. Nenhum valor é impossível, e nenhum é alterado.

## 6. Alvo e base final

In [10]:
base = bruto.replace(config.AUSENTE, np.nan)
base = base.drop_duplicates().reset_index(drop=True)
base[config.TARGET] = (base[config.TARGET] == ">50K").astype(int)

assert list(base.columns) == list(bruto.columns)
assert set(config.FEATURES) | set(config.FORA) | {config.TARGET} == set(base.columns)
assert base[config.TARGET].isin([0, 1]).all()
assert not base.duplicated().any()
assert not (base == config.AUSENTE).any().any()

config.PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
base.to_csv(config.PROCESSED_CSV, index=False)
print("gravado:", config.PROCESSED_CSV.name, base.shape)
print("renda acima de 50 mil:", round(100 * base[config.TARGET].mean(), 1), "%")
print("células vazias por coluna:", base.isna().sum()[lambda s: s > 0].to_dict())

gravado: renda.csv (32537, 15)
renda acima de 50 mil: 24.1 %
células vazias por coluna: {'workclass': 1836, 'occupation': 1843, 'native.country': 582}


**Resultado.**

- **32.537 pessoas e 15 colunas.** O arquivo confere com o hash SHA-256 registrado em `src/config.py`. Saíram 24 linhas repetidas; nenhuma outra linha foi descartada.
- **Valores ausentes ficam, como células vazias:** vínculo (1.836 pessoas), ocupação (1.843) e país de origem (582). Vínculo e ocupação faltam juntos, e quem tem vínculo ausente passa de 50 mil em 10,4% dos casos, contra 24,1% na base: a ausência é informação, e o tratamento fica para o `Pipeline` de cada modelo.
- **12 atributos para os modelos:** 5 numéricos (idade, nível de escolaridade, ganho e perda de capital, horas por semana) e 7 categóricos (vínculo, estado civil, ocupação, papel na família, raça, sexo e país de origem). `education` repete `education.num` em texto, e `fnlwgt` é um peso amostral, não um dado da pessoa; os dois ficam na base, mas fora dos modelos.
- **Alvo:** `income` virou 1 para renda acima de 50 mil. São 24,1% das pessoas, cerca de 3 abaixo para cada 1 acima: a base é mais desbalanceada que a de empréstimos (62,2% de aprovados), e o piso de responder sempre "abaixo de 50 mil" é 75,9%.
- **Para a análise exploratória:** ganho e perda de capital são zero para 91,7% e 95,3% das pessoas, e o ganho tem um teto de 99.999 (159 pessoas, todas acima de 50 mil). São colunas muito assimétricas, o que pesa na padronização que a SVM exige.